# Part 1 — Exploratory Data Analysis
**Dataset**: Ayuraj ASL Kaggle (~8,000 static images, 26 letter classes)

This notebook explores:
1. Class distribution (are classes balanced?)
2. Sample images per class
3. MediaPipe landmark detection rate
4. Feature distribution after normalization

In [ ]:
import os
import sys
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sys.path.insert(0, '../src')
from preprocessing import extract_landmarks

DATA_DIR = '../data/asl_dataset'
sns.set_theme(style='whitegrid')
print('Setup complete')

## 1. Class Distribution

In [ ]:
folders = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
class_counts = {}
for folder in folders:
    files = [f for f in os.listdir(os.path.join(DATA_DIR, folder))
             if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    class_counts[folder] = len(files)

total = sum(class_counts.values())
print(f'Total images: {total}  |  Classes: {len(class_counts)}')
print(f'Min: {min(class_counts.values())}  Max: {max(class_counts.values())}  '
      f'Mean: {np.mean(list(class_counts.values())):.1f}')

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(class_counts.keys(), class_counts.values(), color='steelblue')
ax.set_title('Images per Class', fontsize=14)
ax.set_xlabel('ASL Letter')
ax.set_ylabel('Count')
ax.axhline(np.mean(list(class_counts.values())), color='red', linestyle='--', label='Mean')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Sample Images (2 per class)

In [ ]:
n_cols = 6
n_rows = len(folders)
samples_per_class = 2

fig, axes = plt.subplots(n_rows, samples_per_class * n_cols // n_cols,
                          figsize=(samples_per_class * 3, n_rows * 1.5))

fig, axes = plt.subplots(len(folders), samples_per_class, figsize=(samples_per_class * 2.5, len(folders) * 2))

for row, folder in enumerate(folders):
    folder_path = os.path.join(DATA_DIR, folder)
    files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    sample_files = files[:samples_per_class]
    for col, fname in enumerate(sample_files):
        img = cv2.imread(os.path.join(folder_path, fname))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[row][col].imshow(img_rgb)
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_title(folder, fontsize=9, fontweight='bold')

plt.suptitle('Sample Images per ASL Letter', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3. MediaPipe Detection Rate per Class

In [ ]:
from tqdm.notebook import tqdm

detection_rates = {}
SAMPLE_SIZE = 50  # check first 50 images per class

for folder in tqdm(folders, desc='Checking detection rate'):
    folder_path = os.path.join(DATA_DIR, folder)
    files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg','.jpeg','.png'))][:SAMPLE_SIZE]
    detected = 0
    for fname in files:
        img = cv2.imread(os.path.join(folder_path, fname))
        if img is not None and extract_landmarks(img) is not None:
            detected += 1
    detection_rates[folder] = detected / len(files) if files else 0

fig, ax = plt.subplots(figsize=(14, 4))
colors = ['green' if v >= 0.9 else 'orange' if v >= 0.7 else 'red' for v in detection_rates.values()]
ax.bar(detection_rates.keys(), detection_rates.values(), color=colors)
ax.axhline(0.9, color='green', linestyle='--', alpha=0.5, label='>90%')
ax.set_title(f'MediaPipe Detection Rate per Class (sample={SAMPLE_SIZE})', fontsize=13)
ax.set_xlabel('ASL Letter')
ax.set_ylabel('Detection Rate')
ax.set_ylim(0, 1.1)
ax.legend()
plt.tight_layout()
plt.show()

avg = np.mean(list(detection_rates.values()))
print(f'Average detection rate: {avg:.3f}')
hard = {k: v for k, v in detection_rates.items() if v < 0.7}
if hard:
    print(f'Low detection classes (<70%): {hard}')

## 4. Feature Distribution (after preprocessing)

In [ ]:
import os
X = np.load('../data/X.npy')  # run preprocessing.py first
y = np.load('../data/y.npy')
label_map = np.load('../data/label_map.npy', allow_pickle=True).item()

print(f'X shape: {X.shape}  y shape: {y.shape}')
print(f'Feature range: [{X.min():.3f}, {X.max():.3f}]')
print(f'Feature mean: {X.mean():.4f}  std: {X.std():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Overall feature value distribution
axes[0].hist(X.flatten(), bins=80, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of All Feature Values')
axes[0].set_xlabel('Normalized Landmark Value')
axes[0].set_ylabel('Count')

# Per-class sample count
class_counts_final = Counter(y)
axes[1].bar([label_map[k] for k in sorted(class_counts_final)],
            [class_counts_final[k] for k in sorted(class_counts_final)],
            color='coral')
axes[1].set_title('Samples per Class (after MediaPipe filtering)')
axes[1].set_xlabel('ASL Letter')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()